# 🏆 Game of the Day (GOTD)
Fetches today's games from ESPN across multiple leagues and recommends the best games to watch.

In [13]:
import duckdb
import requests
from datetime import datetime
import pandas as pd

## ⚙️ Configuration

In [14]:
# ── Leagues to fetch ──────────────────────────────────────────────────────────
LEAGUES = [
    ('basketball', 'nba',                     'NBA'),
    ('football',   'nfl',                     'NFL'),
    ('baseball',   'mlb',                     'MLB'),
    ('hockey',     'nhl',                     'NHL'),
    ('basketball', 'mens-college-basketball', 'College Basketball'),
    ('football',   'college-football',        'College Football'),
    ('soccer',     'esp.1',                   'ESP.1'),
    ('soccer',     'eng.1',                   'ENG.1'),
    ('soccer',     'ger.1',                   'GER.1'),
    ('soccer',     'fra.1',                   'FRA.1'),
    ('soccer',     'ita.1',                   'ITA.1'),
]

# ── Average game durations (hours) ───────────────────────────────────────────
DURATION_MAP = {
    'NBA': 2.5, 'NFL': 3.25, 'MLB': 3.0, 'NHL': 2.5,
    'College Basketball': 2.5, 'College Football': 3.5,
    'ESP.1': 2.0, 'ENG.1': 2.0, 'GER.1': 2.0, 'FRA.1': 2.0, 'ITA.1': 2.0
}

# ── Score weights ─────────────────────────────────────────────────────────────
LEAGUE_POINTS = {
    'NFL': 20, 'NBA': 18, 'MLB': 16, 'ESP.1': 14, 'NHL': 12,
    'ENG.1': 10, 'College Basketball': 8, 'College Football': 6,
    'GER.1': 4, 'FRA.1': 3, 'ITA.1': 2
}
FAVORITE_BONUS   = 30
PLAYOFF_BONUS    = 25
DERBY_BONUS      = 15
STREAK_BONUS     = 15
MAX_TABLE_POINTS = 10
TIME_CUTOFF      = '02:00'
TOP_N            = 5

# ── Personal preferences ─────────────────────────────────────────────────────
FAVORITE_TEAMS = [
    'Miami Heat', 'Carolina Panthers',
    'Illinois Fighting Illini', 'San Diego Padres', 'Valencia'
]

DERBIES = [
    ('Los Angeles Lakers',  'Los Angeles Clippers'),
    ('New York Knicks',     'Brooklyn Nets'),
    ('Real Madrid',         'Barcelona'),
    ('AC Milan',            'Internazionale'),
    ('Bayern Munich',       'Borussia Dortmund'),
    ('Arsenal',             'Tottenham Hotspur'),
    ('Manchester United',   'Manchester City'),
    ('Chicago Bulls',       'Chicago Bears'),   # same city different sport example
    # Add more derbies here
]

## 📡 Data Fetching

In [15]:
def get_record(competitor):
    """Extract win-loss record from a competitor object."""
    records = competitor.get('records', [])
    return records[0].get('summary', '0-0') if records else 'N/A'


def calculate_end_time(dt, league):
    """Calculate estimated end time based on league duration."""
    hours = DURATION_MAP.get(league, 2.5)
    return (dt + pd.Timedelta(hours=hours)).strftime('%H:%M')


def fetch_league(sport, league_slug, league_name):
    """Fetch today's games for a single league from ESPN API."""
    today = datetime.now().strftime('%Y%m%d')
    url = f"https://site.api.espn.com/apis/site/v2/sports/{sport}/{league_slug}/scoreboard"
    params = {'dates': today}

    if league_slug == 'mens-college-basketball':
        params.update({'groups': 50, 'limit': 350})
    elif league_slug == 'college-football':
        params['groups'] = 80

    try:
        data = requests.get(url, params=params).json()
        games = []

        for event in data.get('events', []):
            comp = event['competitions'][0]
            dt_cet = pd.to_datetime(event.get('date')).tz_convert('Europe/Vienna')

            # Date filter for college leagues (API sometimes returns extra dates)
            if league_slug in ('mens-college-basketball', 'college-football'):
                if dt_cet.strftime('%Y%m%d') != today:
                    continue

            home = next(c for c in comp['competitors'] if c['homeAway'] == 'home')
            away = next(c for c in comp['competitors'] if c['homeAway'] == 'away')

            # Check if game is a playoff/finals game
            notes = comp.get('notes', [])
            is_playoff = any(
                'playoff' in str(n.get('headline', '')).lower() or
                'final' in str(n.get('headline', '')).lower()
                for n in notes
            )

            games.append({
                'Away Team':      away['team']['displayName'],
                'Away Record':    get_record(away),
                'Home Team':      home['team']['displayName'],
                'Home Record':    get_record(home),
                'Date':           dt_cet.strftime('%Y-%m-%d'),
                'Time (CET)':     dt_cet.strftime('%H:%M'),
                'End Time (CET)': calculate_end_time(dt_cet, league_name),
                'League':         league_name,
                'Is Playoff':     is_playoff,
                'Watched':        0,
            })

        return pd.DataFrame(games)

    except Exception as e:
        print(f"Error fetching {league_name}: {e}")
        return pd.DataFrame()


def fetch_all_leagues():
    """Fetch today's games for all configured leagues."""
    dfs = [fetch_league(sport, slug, name) for sport, slug, name in LEAGUES]
    return pd.concat(dfs, ignore_index=True)


# Run
all_games = fetch_all_leagues()
print(f"✅ {len(all_games)} games fetched across {all_games['League'].nunique()} leagues")
all_games

✅ 36 games fetched across 7 leagues


,Away Team,Away Record,Home Team,Home Record,Date,Time (CET),End Time (CET),League,Is Playoff,Watched
0,Detroit Pistons,58-22,Charlotte Hornets,43-37,2026-04-11,01:00,03:30,NBA,False,0
1,Miami Heat,41-39,Washington Wizards,17-63,2026-04-11,01:00,03:30,NBA,False,0
2,Cleveland Cavaliers,51-29,Atlanta Hawks,45-35,2026-04-11,01:00,03:30,NBA,False,0
3,New Orleans Pelicans,26-54,Boston Celtics,54-26,2026-04-11,01:30,04:00,NBA,False,0
4,Philadelphia 76ers,43-37,Indiana Pacers,19-61,2026-04-11,01:30,04:00,NBA,False,0
5,Toronto Raptors,45-35,New York Knicks,52-28,2026-04-11,01:30,04:00,NBA,False,0
6,Orlando Magic,44-36,Chicago Bulls,31-49,2026-04-11,02:00,04:30,NBA,False,0
7,Brooklyn Nets,20-60,Milwaukee Bucks,31-49,2026-04-11,02:00,04:30,NBA,False,0
8,Dallas Mavericks,25-55,San Antonio Spurs,61-19,2026-04-11,02:00,04:30,NBA,False,0
9,Oklahoma City Thunder,64-16,Denver Nuggets,52-28,2026-04-11,03:00,05:30,NBA,False,0


## 🧮 Scoring Algorithm

In [16]:
def parse_record(record_str):
    """Extract wins and losses from a record string like '12-5' or '14-2-4'."""
    try:
        parts = str(record_str).split('-')
        return int(parts[0]), int(parts[1])
    except:
        return 0, 0


def get_win_pct(record_str):
    """Calculate win percentage from a record string."""
    wins, losses = parse_record(record_str)
    total = wins + losses
    return wins / total if total > 0 else 0


def is_derby(home, away):
    """Check if a game is a derby."""
    return any(
        (home == t1 and away == t2) or (home == t2 and away == t1)
        for t1, t2 in DERBIES
    )


#def is_after_cutoff(time_str):
 #   """Return True if game starts after the cutoff time."""
  #  try:
   #     return datetime.strptime(time_str, '%H:%M') >= datetime.strptime(TIME_CUTOFF, '%H:%M')
    #except:
     #   return False


def has_winning_streak(record_str, threshold=5):
    """Simple streak detection: high win % with enough games played."""
    wins, losses = parse_record(record_str)
    total = wins + losses
    return total >= threshold and (wins / total) > 0.75


#def calculate_score(row):
 #   """Calculate a watchability score for a single game."""
  #  if is_after_cutoff(row['Time (CET)']):
   #     return None

    #score = 0
    #home, away = row['Home Team'], row['Away Team']

    # Liga-Punkte
    #score += LEAGUE_POINTS.get(row['League'], 0)

    # Lieblingsmannschaft
    #if home in FAVORITE_TEAMS or away in FAVORITE_TEAMS:
     #   score += FAVORITE_BONUS

    # Playoff / Finals
    #if row.get('Is Playoff', False):
     #   score += PLAYOFF_BONUS

    # Derby
    #if is_derby(home, away):
     #   score += DERBY_BONUS

    # Tabellenposition (Durchschnitt beider Teams, max 10 Punkte)
    #avg_pct = (get_win_pct(row['Home Record']) + get_win_pct(row['Away Record'])) / 2
    #score += round(avg_pct * MAX_TABLE_POINTS)

    # Winning Streak
    #if has_winning_streak(row['Home Record']) or has_winning_streak(row['Away Record']):
     #   score += STREAK_BONUS

#    return score

## 🏅 Top Games of the Day

In [17]:
#def get_top_games(df, top_n=TOP_N):
  #  """Score all games and return the top N recommendations."""
   # df = df.copy()

    # Calculate scores
    #df['Score'] = df.apply(calculate_score, axis=1)
   # Remove games after cutoff
#    df = df[df['Score'].notna()].copy()

    # Flag favorite team games
 #   df['Is Favorite'] = df.apply(
  #      lambda r: r['Home Team'] in FAVORITE_TEAMS or r['Away Team'] in FAVORITE_TEAMS,
   #     axis=1
    #)

    # Sort: favorites first, then by score
    #df = df.sort_values(by=['Is Favorite', 'Score'], ascending=[False, False])

    #top = df.head(top_n)

    # Pretty print
    #print(f"\n🏆 TOP {top_n} SPIELE DES TAGES\n" + "="*50)
    #for i, (_, row) in enumerate(top.iterrows(), 1):
     #   tags = []
      #  if row['Is Favorite']:          tags.append('⭐ Favorit')
       # if row.get('Is Playoff'):       tags.append('🏆 Playoff')
        #if is_derby(row['Home Team'], row['Away Team']): tags.append('🔥 Derby')
        #tag_str = '  ' + ' | '.join(tags) if tags else ''

        #print(f"\n#{i}{tag_str}")
        #print(f"   {row['Away Team']} @ {row['Home Team']}")
        #print(f"   Liga:    {row['League']}")
        #print(f"   Zeit:    {row['Time (CET)']} – {row['End Time (CET)']}")
        #print(f"   Records: {row['Away Record']} vs {row['Home Record']}")
        #print(f"   Score:   {int(row['Score'])}")

    #return top


# Run
#top_games = get_top_games(all_games)

## 💾 Export

In [18]:
# Mark watched games (set index of watched game to 1)
# Example: all_games.at[10, 'Watched'] = 1

# Export to CSV
#today_str = datetime.now().strftime('%Y%m%d')
#filename = f'games_{today_str}.csv'
#all_games.to_csv(filename, index=False)
#print(f"✅ Saved to {filename}")


In [19]:
#export to dbt
# --- 1. Prepare helper columns for dbt ---
# We label the games so dbt knows which ones to give bonus points to
all_games['is_favorite'] = all_games.apply(
    lambda r: 1 if r['Home Team'] in FAVORITE_TEAMS or r['Away Team'] in FAVORITE_TEAMS else 0, axis=1
)

all_games['is_derby_game'] = all_games.apply(
    lambda r: 1 if is_derby(r['Home Team'], r['Away Team']) else 0, axis=1
)

all_games['has_streak'] = all_games.apply(
    lambda r: 1 if has_winning_streak(r['Home Record']) or has_winning_streak(r['Away Record']) else 0, axis=1
)

# We do the "hard" math of win percentage here
all_games['home_win_pct'] = all_games['Home Record'].apply(get_win_pct)
all_games['away_win_pct'] = all_games['Away Record'].apply(get_win_pct)
# Connect to the DuckDB file
con = duckdb.connect('sports.duckdb')

# Register your pandas DataFrame 'all_games' so DuckDB can see it
con.register('df_view', all_games)

# Create a permanent table from that view
con.execute("CREATE OR REPLACE TABLE raw_games AS SELECT * FROM df_view")

print("✅ Data successfully saved to sports.duckdb!")
con.close()

✅ Data successfully saved to sports.duckdb!


In [20]:
# Connect to the database file
con = duckdb.connect('sports.duckdb')

# Ask DuckDB for the top games from the table dbt created
df_leaderboard = con.execute("""
    SELECT 
        total_watch_score,
        tags,
        "League",
        "Away Team", 
        "Home Team", 
        "Time (CET)"
    FROM fct_daily_schedule
    ORDER BY total_watch_score DESC
""").df()

con.close()

# Show the results beautifully
print("🏆 YOUR TOP GAMES FOR TODAY 🏆")
df_leaderboard.head(10)

🏆 YOUR TOP GAMES FOR TODAY 🏆


,total_watch_score,tags,League,Away Team,Home Team,Time (CET)
0,52.0,Favorite,NBA,Miami Heat,Washington Wizards,01:00
1,24.0,,NBA,Detroit Pistons,Charlotte Hornets,01:00
2,24.0,,NBA,Cleveland Cavaliers,Atlanta Hawks,01:00
3,24.0,,NBA,Toronto Raptors,New York Knicks,01:30
4,23.0,,NBA,New Orleans Pelicans,Boston Celtics,01:30
5,22.0,,NBA,Philadelphia 76ers,Indiana Pacers,01:30
6,22.0,,MLB,Cleveland Guardians,Atlanta Braves,01:15
7,21.0,,MLB,Arizona Diamondbacks,Philadelphia Phillies,00:40
8,21.0,,MLB,Miami Marlins,Detroit Tigers,00:40
9,21.0,,MLB,Los Angeles Angels,Cincinnati Reds,00:45


In [24]:
con = duckdb.connect('sports.duckdb')

# Drop the old one if it exists so we can add the 'Unique' constraint
con.execute("DROP TABLE IF EXISTS watched_history")

# Create the permanent archive with a unique constraint
con.execute("""
    CREATE TABLE watched_history (
        event_date DATE,
        league TEXT,
        matchup TEXT,
        score FLOAT,
        watched BOOLEAN DEFAULT FALSE,
        UNIQUE(event_date, matchup) 
    )
""")
con.close()

In [29]:
con = duckdb.connect('sports.duckdb')

# 2. Query the permanent history table
# We use ORDER BY event_date DESC to put the newest days at the top
df_history = con.execute("""
    SELECT 
        CAST(score AS INT) as score,
        event_date as date,
        league,
        matchup,
        watched
    FROM watched_history
    ORDER BY event_date DESC, score DESC
""").df()

con.close()

# 3. Display
print("📖 GLOBAL WATCH HISTORY 📖")
display(df_history)

📖 GLOBAL WATCH HISTORY 📖


,score,date,league,matchup,watched
0,52,2026-04-10,NBA,Miami Heat @ Washington Wizards,False
1,24,2026-04-10,NBA,Toronto Raptors @ New York Knicks,False
2,24,2026-04-10,NBA,Cleveland Cavaliers @ Atlanta Hawks,False
3,24,2026-04-10,NBA,Detroit Pistons @ Charlotte Hornets,False
4,23,2026-04-10,NBA,New Orleans Pelicans @ Boston Celtics,False
5,22,2026-04-10,MLB,Cleveland Guardians @ Atlanta Braves,False
6,22,2026-04-10,NBA,Philadelphia 76ers @ Indiana Pacers,False
7,21,2026-04-10,MLB,Athletics @ New York Mets,False
8,21,2026-04-10,MLB,New York Yankees @ Tampa Bay Rays,False
9,21,2026-04-10,MLB,Washington Nationals @ Milwaukee Brewers,False


In [33]:
import duckdb
con = duckdb.connect('sports.duckdb')

# 1. Clear the old structure
con.execute("DROP TABLE IF EXISTS watched_history")

# 2. Create the complete structure
con.execute("""
    CREATE TABLE watched_history (
        event_date DATE,
        league TEXT,
        matchup TEXT,
        score FLOAT,
        "Time (CET)" TEXT, 
        watched BOOLEAN DEFAULT FALSE,
        UNIQUE(event_date, matchup)
    )
""")
con.close()
print("✅ History table reset! It is now ready for the Time column.")

✅ History table reset! It is now ready for the Time column.


In [41]:
import duckdb
con = duckdb.connect('sports.duckdb')

df_history = con.execute("""
    SELECT 
        CAST(score AS INT) as score,
        event_date as date,
        league,
        matchup,
        time,
        watched
    FROM watched_history
    ORDER BY event_date DESC, score DESC
""").df()

con.close()
display(df_history)

,score,date,league,matchup,time,watched
0,52,2026-04-10,NBA,Miami Heat @ Washington Wizards,01:00,False
1,36,2026-04-10,ESP.1,Girona @ Real Madrid,21:00,False
2,24,2026-04-10,NBA,Detroit Pistons @ Charlotte Hornets,01:00,False
3,24,2026-04-10,NBA,Toronto Raptors @ New York Knicks,01:30,False
4,24,2026-04-10,FRA.1,AS Monaco @ Paris FC,19:00,False
5,24,2026-04-10,NBA,Cleveland Cavaliers @ Atlanta Hawks,01:00,False
6,24,2026-04-10,FRA.1,Metz @ Marseille,21:05,False
7,23,2026-04-10,NBA,New Orleans Pelicans @ Boston Celtics,01:30,False
8,22,2026-04-10,NBA,Philadelphia 76ers @ Indiana Pacers,01:30,False
9,22,2026-04-10,MLB,Cleveland Guardians @ Atlanta Braves,01:15,False


In [ ]:
### harlequin sports.duckdb

In [ ]:
###harlequin query for watched history

SELECT  * FROM v_history ORDER BY date DESC, score DESC;

In [ ]:
#update statement to mark a game as watched in harlequin

UPDATE watched_history 
SET watched = 1 
WHERE matchup = 'Away Team @ Home Team' 
AND event_date = '2026-04-10';